**Cortex Code CLI Prompt**

*Create a notebook in the same workspace (My Workspace). The notebook should demonstrate how to run a simple search via. SQL, the Python API and the REST API. An example of a simple search is a vector search on just the SUMMARY column. After the simple search examples, the notebook should include more complicated search examples via. the Python API. Complicated examples could include searching multiple vector feilds and key word feilds and setting the scoring_config to boast more recent restults (time decay).*

# Cortex Search Demonstration

This notebook provides examples querying the Cortex Search service `NEWS_ARTICLES_SEARCH_INDEX`. 

The `NEWS_ARTICLES_SEARCH_INDEX` provides search on the `FSI_DEMO_DB.CAPITAL_MARKETS.NEWS_ARTICLES` table. The search index configuration includes vector and key word search fields and exact match filter fields.

The vector search fields include the HEADLINE, CONTENT, SUMMARY columns from the `NEWS_ARTICLES` table. These fields are embedded via. the `snowflake-arctic-embed-l-v2.0` model.

The keyword search fields include the TICKERS_MENTIONED column from the `NEWS_ARTICLES` table.

All of the other columns from the table (ARTICLE_ID, PUBLISH_DATE, SOURCE, AUTHOR, URL, SECTORS_MENTIONED, ARTICLE_TYPE, SENTIMENT_SCORE, SENTIMENT_LABEL, RELEVANCE_SCORE, CREATED_AT) can be filtered on.

This notebook, has two sections 

1. Simple Search Examples via. 
   - SQL
   - Python API
   - REST API
   
2. Advanced Search Examples
   - Multiple field search
   - Combine vector and keyword search
   - Time decay scoring for recent results

# Setup

In [ ]:
# Import required libraries
from snowflake.snowpark.context import get_active_session
from snowflake.core import Root
import json

# Get the active Snowflake session (works in Snowflake Notebooks)
session = get_active_session()
root = Root(session)

# Get reference to our Cortex Search service
search_service = (
    root
    .databases["FSI_DEMO_DB"]
    .schemas["CAPITAL_MARKETS"]
    .cortex_search_services["NEWS_ARTICLES_SEARCH_INDEX"]
)

print("Setup complete! Search service ready.")

# 1. Simple Search Examples

These examples demonstrate basic vector search on the SUMMARY column using three different methods.

## Search via SQL

The `SNOWFLAKE.CORTEX.SEARCH_PREVIEW` function allows you to query a Cortex Search service directly from SQL.

In [ ]:
# Simple vector search on SUMMARY column via SQL, add the parse JSON function to cleanly display the output
sql_query = """
SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'FSI_DEMO_DB.CAPITAL_MARKETS.NEWS_ARTICLES_SEARCH_INDEX',
        '{
            "query": "Federal Reserve interest rate decision impact on markets",
            "columns": ["ARTICLE_ID", "HEADLINE", "SUMMARY", "PUBLISH_DATE", "SOURCE"],
            "limit": 5
        }'
    )
)['results'] AS results
"""

results_df = session.sql(sql_query).collect()
print("SQL Search Results:")
print(json.dumps(json.loads(results_df[0]['RESULTS']), indent=2))

## Search via Python API

The Python API provides a more programmatic way to query the search service with native Python objects.

In [ ]:
# Simple vector search on SUMMARY column via Python API
response = search_service.search(
    query="Federal Reserve interest rate decision impact on markets",
    columns=["ARTICLE_ID", "HEADLINE", "SUMMARY", "PUBLISH_DATE", "SOURCE"],
    limit=5
)

print("Python API Search Results:")
print(json.dumps(response.to_dict(), indent=2))

## Search via REST API

The REST API allows external applications to query the Cortex Search service. Below is an example using Python's `requests` library.

This example uses a Programmatic Access Token (PAT) for authentication. You'll need to generate a PAT in Snowsight or use another authentication method.

In [ ]:
import requests

# Configuration - Update these values for your environment
ACCOUNT_URL = "<your_account>.snowflakecomputing.com"  # e.g., "abc12345.us-east-1.snowflakecomputing.com"
PAT_TOKEN = "<your_programmatic_access_token>"  # Generate in Snowsight under User Menu > Programmatic Access Tokens

# REST API endpoint for our search service
endpoint = f"https://{ACCOUNT_URL}/api/v2/databases/FSI_DEMO_DB/schemas/CAPITAL_MARKETS/cortex-search-services/NEWS_ARTICLES_SEARCH_INDEX:query"

# Request headers
headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "Authorization": f"Bearer {PAT_TOKEN}"
}

# Request body - Simple vector search
payload = {
    "query": "Federal Reserve interest rate decision impact on markets",
    "columns": ["ARTICLE_ID", "HEADLINE", "SUMMARY", "PUBLISH_DATE", "SOURCE"],
    "limit": 5
}

# Make the request (uncomment to execute - requires valid credentials)
# response = requests.post(endpoint, headers=headers, json=payload)
# print("REST API Search Results:")
# print(json.dumps(response.json(), indent=2))

print("REST API Example (requires valid ACCOUNT_URL and PAT_TOKEN):")
print(f"Endpoint: {endpoint}")
print(f"Payload: {json.dumps(payload, indent=2)}")

### cURL Example for REST API

You can also query the REST API using cURL from the command line:

```bash
curl --location 'https://<account_url>/api/v2/databases/FSI_DEMO_DB/schemas/CAPITAL_MARKETS/cortex-search-services/NEWS_ARTICLES_SEARCH_INDEX:query' \
  --header 'Content-Type: application/json' \
  --header 'Accept: application/json' \
  --header 'Authorization: Bearer <your_pat_token>' \
  --data '{
    "query": "Federal Reserve interest rate decision impact on markets",
    "columns": ["ARTICLE_ID", "HEADLINE", "SUMMARY", "PUBLISH_DATE", "SOURCE"],
    "limit": 5
  }'
```

# 2. Advanced Search Examples

These example demonstrate more advanced search options including: multiple field search, combine vector and keyword search, time decay scoring for recent results

## Search with Filters

Filter results based on attribute columns while performing vector search.

In [ ]:
# Search with filters - only positive sentiment articles from specific sources
response = search_service.search(
    query="technology sector earnings growth",
    columns=["ARTICLE_ID", "HEADLINE", "SUMMARY", "SOURCE", "SENTIMENT_LABEL", "PUBLISH_DATE"],
    filter={
        "@and": [
            {"@eq": {"SENTIMENT_LABEL": "positive"}},
            {"@or": [
                {"@eq": {"SOURCE": "Reuters"}},
                {"@eq": {"SOURCE": "Bloomberg"}}
            ]}
        ]
    },
    limit=5
)

print("Filtered Search Results (Positive sentiment from Reuters/Bloomberg):")
print(json.dumps(response.to_dict(), indent=2))

## Multi-feild search with Boasting

Search across multiple feilds, boast certain feild for better search relevancy

In [ ]:
# Search across HEADLINE, CONTENT, and SUMMARY with vector boosts
response = search_service.search(
    multi_index_query={
        # Search the same query across multiple vector indices
        "HEADLINE": [{"text": "inflation economic outlook"}],
        "CONTENT": [{"text": "inflation economic outlook"}],
        "SUMMARY": [{"text": "inflation economic outlook"}]
    },
    columns=["ARTICLE_ID", "HEADLINE", "SUMMARY", "CONTENT", "PUBLISH_DATE", "SOURCE"],
    scoring_config={
        # Adjust weights for text vs vector vs reranker scoring
        "weights": {
            "texts": 1,
            "vectors": 3,  # Emphasize semantic similarity
            "reranker": 2
        },
        "functions": {
            # Boost specific vector indices - prioritize HEADLINE matches
            "vector_boosts": [
                {"column": "HEADLINE", "weight": 3},  # Headlines are most important
                {"column": "SUMMARY", "weight": 2},   # Summaries are second
                {"column": "CONTENT", "weight": 1}    # Full content has lowest weight
            ]
        }
    },
    limit=5
)

print("Multi-Index Vector Search Results (HEADLINE > SUMMARY > CONTENT weighting):")
print(json.dumps(response.to_dict(), indent=2))

## Combined Vector and Keyword Search

Search using both vector indices (semantic search) and text indices (keyword search) simultaneously.

In [ ]:
# Combined vector search on SUMMARY + keyword search on TICKERS_MENTIONED
response = search_service.search(
    multi_index_query={
        # Semantic search on SUMMARY for context
        "SUMMARY": [{"text": "quarterly earnings report beat expectations"}],
        # Keyword search on TICKERS_MENTIONED for specific stocks
        "TICKERS_MENTIONED": [{"text": "AAPL MSFT GOOGL"}]
    },
    columns=["ARTICLE_ID", "HEADLINE", "SUMMARY", "TICKERS_MENTIONED", "PUBLISH_DATE", "SENTIMENT_LABEL"],
    scoring_config={
        "weights": {
            "texts": 2,    # Give keyword matches good weight
            "vectors": 2,  # Balance with semantic similarity
            "reranker": 1
        },
        "functions": {
            # Boost the text index for ticker matching
            "text_boosts": [
                {"column": "TICKERS_MENTIONED", "weight": 2}
            ],
            "vector_boosts": [
                {"column": "SUMMARY", "weight": 1}
            ]
        }
    },
    limit=5
)

print("Combined Vector + Keyword Search Results:")
print(json.dumps(response.to_dict(), indent=2))

## Time Decay Scoring: Boost Recent Results

Use `time_decays` in the scoring configuration to boost more recent articles. This is useful for news search where recency matters.

In [ ]:
# Search with time decay - boost articles published recently
response = search_service.search(
    query="market volatility trading strategies",
    columns=["ARTICLE_ID", "HEADLINE", "SUMMARY", "PUBLISH_DATE", "SOURCE", "SENTIMENT_LABEL"],
    scoring_config={
        "functions": {
            "time_decays": [
                {
                    "column": "PUBLISH_DATE",  # Use PUBLISH_DATE for recency
                    "weight": 2,                # Higher weight = stronger recency preference
                    "limit_hours": 168          # 7 days - articles older than this get minimal boost
                }
            ]
        }
    },
    limit=10
)

print("Time Decay Search Results (Recent articles boosted, 7-day window):")
print(json.dumps(response.to_dict(), indent=2))

## Multi-feild + Time Decay + Numeric Boost

Combine all advanced features: search multiple fields, boost recent articles, and boost by relevance score.

In [ ]:
# Advanced search combining multiple features
response = search_service.search(
    multi_index_query={
        "HEADLINE": [{"text": "stock market analysis investment opportunities"}],
        "SUMMARY": [{"text": "stock market analysis investment opportunities"}],
        "TICKERS_MENTIONED": [{"text": "SPY QQQ"}]  # ETF tickers
    },
    columns=[
        "ARTICLE_ID", "HEADLINE", "SUMMARY", "TICKERS_MENTIONED",
        "PUBLISH_DATE", "SOURCE", "RELEVANCE_SCORE", "SENTIMENT_LABEL"
    ],
    filter={
        "@or": [
            {"@eq": {"SENTIMENT_LABEL": "positive"}},
            {"@eq": {"SENTIMENT_LABEL": "neutral"}}
        ]
    },
    scoring_config={
        "weights": {
            "texts": 1,
            "vectors": 2,
            "reranker": 1
        },
        "functions": {
            # Boost by index importance
            "vector_boosts": [
                {"column": "HEADLINE", "weight": 2},
                {"column": "SUMMARY", "weight": 1}
            ],
            "text_boosts": [
                {"column": "TICKERS_MENTIONED", "weight": 1.5}
            ],
            # Boost recent articles
            "time_decays": [
                {
                    "column": "PUBLISH_DATE",
                    "weight": 1.5,
                    "limit_hours": 720  # 30 days
                }
            ],
            # Boost articles with higher relevance scores
            "numeric_boosts": [
                {
                    "column": "RELEVANCE_SCORE",
                    "weight": 1
                }
            ]
        }
    },
    limit=10
)

print("Advanced Combined Search Results:")
print("(Multi-index + Time decay + Numeric boost + Filters)")
print(json.dumps(response.to_dict(), indent=2))

## Disable Reranker for Lower Latency

For time-sensitive applications, you can disable the reranker to reduce latency (100-300ms savings).

In [ ]:
# Search with reranker disabled for faster response
response = search_service.search(
    query="breaking news financial markets",
    columns=["ARTICLE_ID", "HEADLINE", "SUMMARY", "PUBLISH_DATE"],
    scoring_config={
        "reranker": "none"  # Disable reranker for lower latency
    },
    limit=5
)

print("Low-Latency Search Results (Reranker Disabled):")
print(json.dumps(response.to_dict(), indent=2))

## Custom Component Weights

Adjust the relative importance of text matching, vector similarity, and reranking in the final score.

In [ ]:
# Search emphasizing semantic similarity over keyword matching
response = search_service.search(
    query="economic recession indicators warning signs",
    columns=["ARTICLE_ID", "HEADLINE", "SUMMARY", "PUBLISH_DATE", "SECTORS_MENTIONED"],
    scoring_config={
        "functions": {
            "weights": {
                "texts": 1,      # Low weight for exact keyword matches
                "vectors": 5,    # High weight for semantic similarity
                "reranker": 3    # Medium weight for reranking
            }
        }
    },
    limit=5
)

print("Semantic-Focused Search Results (High vector weight):")
print(json.dumps(response.to_dict(), indent=2))